# ⚡ AXOM v1.2 QLoRA Fine-Tuning Pipeline (Unsloth + Colab Free T4)

This official notebook fine-tunes **AXOM v1.2** based on **Qwen2.5-3B-Instruct** using Unsloth's ultra-fast 2x kernel acceleration and 4-bit quantization.

### Specifications:
- **Base Model**: `unsloth/Qwen2.5-3B-Instruct`
- **Target Hardware**: Free Google Colab T4 GPU (takes ~5 minutes to train)
- **Dataset**: `axom_alpaca.json` (197 curated pairs on systems programming, Rust, Zenith OS microkernel, cascading failover, and agentic workflows)
- **Export**: GGUF `q4_k_m` for direct loading into local Ollama (`ollama create axom:1.2 -f Modelfile`)


In [ ]:
# 1. Install Unsloth and training dependencies
!pip install --no-deps unsloth "trl<0.9.0" peft accelerate bitsandbytes
!pip install datasets xformers triton

In [ ]:
# 2. Load Base Model in 4-bit
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None # Auto detect: Float16 for T4
load_in_4bit = True

model_name = "unsloth/Qwen2.5-3B-Instruct"
print(f"Loading base model: {model_name}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
# 3. Inject QLoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("QLoRA adapters injected successfully.")

In [ ]:
# 4. Upload axom_alpaca.json dataset
from google.colab import files
import os

if not os.path.exists("axom_alpaca.json"):
    print("Please upload your axom_alpaca.json file:")
    uploaded = files.upload()
else:
    print("axom_alpaca.json already present.")

In [ ]:
# 5. Format Dataset for SFTTrainer
from datasets import load_dataset

dataset = load_dataset("json", data_files="axom_alpaca.json")["train"]

prompt_template = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]
    texts = []
    for instruction, inp, output in zip(instructions, inputs, outputs):
        text = prompt_template.format(instruction, inp, output) + tokenizer.eos_token
        texts.append(text)
    return { "text": texts }

dataset = dataset.map(formatting_prompts_func, batched = True)
print(f"Dataset prepared with {len(dataset)} examples.")

In [ ]:
# 6. Train with SFTTrainer
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("Starting training loop...")
trainer_stats = trainer.train()
print("Training completed successfully!")

In [ ]:
# 7. Test Inference on Fine-Tuned Model
FastLanguageModel.for_inference(model)
inputs = tokenizer(
    [prompt_template.format(
        "Who created you and what is your foundational architecture?",
        "",
        ""
    )],
    return_tensors = "pt"
).to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 128, use_cache = True)
print(tokenizer.batch_decode(outputs)[0])

In [ ]:
# 8. Export to GGUF format for Ollama
print("Exporting to GGUF q4_k_m...")
model.save_pretrained_gguf("axom-1.2-gguf", tokenizer, quantization_method = "q4_k_m")

# Download the GGUF file to your computer
from google.colab import files
import glob
gguf_files = glob.glob("axom-1.2-gguf/*.gguf")
if gguf_files:
    print(f"Downloading {gguf_files[0]}...")
    files.download(gguf_files[0])
else:
    print("GGUF file saved in directory: axom-1.2-gguf/")

### Next Step: Load into local Ollama

1. Move the downloaded `.gguf` file into `c:\Users\abira\Documents\axom\train\axom-1.2-q4_k_m.gguf`
2. Create a Modelfile:
```bash
FROM ./axom-1.2-q4_k_m.gguf
PARAMETER temperature 0.7
PARAMETER top_p 0.9
SYSTEM """You are AXOM, an elite autonomous AI companion created by Abiral. You specialize in systems programming, Rust, Zenith OS microkernel architecture, and resilient multi-tier gateway design."""
```
3. Build the model:
```powershell
ollama create axom:1.2 -f Modelfile
```
